# 09 - Build the Tableau workbook

**Goal:** generate `dashboard/tableau/retail_dashboard_generated.twb`, a Tableau workbook with five dashboards that mirror the HTML dashboard, connected to the CSV files in `dashboard/tableau/data/` (written by notebook 08).

**How:** a `.twb` is XML. The structure here follows the sample workbook that ships with Tableau Desktop (`Performance/PerformanceRecording_new.twb`): text-file data sources, worksheets (a view definition plus pane marks and encodings) and dashboards (zones that place worksheets on a canvas). The XML is built with `xml.etree.ElementTree`, so it is always well-formed.

**Design choices that keep the XML simple**
- Filters are applied in the data files (full months only, cohort months 1-12), so the workbook has no filter definitions.
- Stacked and grouped charts read long-format files, so no Measure Names / Measure Values.
- Tables are pre-formatted text, shown as dimension lists with a hidden rank column for ordering.
- Chart legends are written into the sheet titles as coloured text, using Tableau's default blue and orange (Tableau Desktop Free Edition ignores custom colour palettes, so the marks keep Tableau's defaults).

Open the result in Tableau Desktop (double-click the file). Tableau stores absolute paths to the CSVs; if you move the project, use Data > Edit Data Source > Replace.

**Output names end in `_generated`.** `dashboard/tableau/retail_dashboard.twbx` is the workbook that was opened and re-saved in Tableau (it contains edits made by hand), so this notebook never writes to it. Re-running this notebook only rewrites `retail_dashboard_generated.twb` and `retail_dashboard_generated.twbx`. To use the generated version, rename it or copy any hand edits across.

In [1]:
import pandas as pd
import xml.etree.ElementTree as ET
from pathlib import Path

ROOT = Path.cwd().parent  # notebook runs from notebooks/
DATA_DIR = ROOT / "dashboard" / "tableau" / "data"
OUT_FILE = ROOT / "dashboard" / "tableau" / "retail_dashboard_generated.twb"
assert OUT_FILE.stem.endswith("_generated"), "never overwrite the workbook that was edited in Tableau"

BLUE, ORANGE = "#4e79a7", "#f28e2b"  # Tableau's default first two colours
INK, INK2 = "#0b0b0b", "#52514e"
DATE_COLUMNS = {"month", "cohort_month"}

def sub(parent, tag, attrs=None, text=None):
    el = ET.SubElement(parent, tag, {k: str(v) for k, v in (attrs or {}).items()})
    if text is not None:
        el.text = text
    return el

## 1. Data sources and fields

In [2]:
class DataSource:
    # One CSV file = one Tableau text-file data source.
    def __init__(self, key, caption, filename):
        self.key, self.caption, self.filename = key, caption, filename
        frame = pd.read_csv(DATA_DIR / filename)
        self.columns = list(frame.columns)
        self.types = {}
        for col in self.columns:
            if col in DATE_COLUMNS:
                self.types[col] = "date"
            elif pd.api.types.is_integer_dtype(frame[col]):
                self.types[col] = "integer"
            elif pd.api.types.is_float_dtype(frame[col]):
                self.types[col] = "real"
            else:
                self.types[col] = "string"

    def element(self):
        ds = ET.Element("datasource", {"caption": self.caption, "inline": "true", "name": self.key, "version": "18.1"})
        conn = sub(ds, "connection", {"class": "federated"})
        named = sub(sub(conn, "named-connections"), "named-connection", {"name": f"{self.key}_leaf"})
        sub(named, "connection", {
            "character-set": "UTF-8", "class": "textscan", "directory": DATA_DIR.as_posix(), "driver": "",
            "filename": self.filename, "force-character-set": "no", "force-header": "no", "force-separator": "no",
            "header": "yes", "separator": ",", "text-qualifier": '"',
        })
        table = self.filename.replace(".", "#")
        rel = sub(conn, "relation", {"connection": f"{self.key}_leaf", "name": table, "table": f"[{table}]", "type": "table"})
        cols = sub(rel, "columns", {"character-set": "UTF-8", "header": "yes", "locale": "en_US", "separator": ",", "text-qualifier": '"'})
        for i, col in enumerate(self.columns):
            sub(cols, "column", {"datatype": self.types[col], "name": col, "ordinal": i})
        sub(conn, "refresh", {"increment-key": "", "incremental-updates": "false"})
        return ds


class Field:
    # A field as used on a shelf: a column plus how it is aggregated or typed.
    KINDS = {
        # kind: (instance prefix, suffix, derivation, instance type, column role, column type)
        "sum":     ("sum", "qk", "Sum", "quantitative", "measure", "quantitative"),
        "avg":     ("avg", "qk", "Avg", "quantitative", "measure", "quantitative"),
        "nom":     ("none", "nk", "None", "nominal", "dimension", "nominal"),
        "ord":     ("none", "ok", "None", "ordinal", "dimension", "ordinal"),
        "cont":    ("none", "qk", "None", "quantitative", "dimension", "quantitative"),
        "month_c": ("tmn", "qk", "Month-Trunc", "quantitative", "dimension", "ordinal"),
        "month_d": ("tmn", "ok", "Month-Trunc", "ordinal", "dimension", "ordinal"),
    }

    def __init__(self, ds, col, kind):
        self.ds, self.col, self.kind = ds, col, kind
        prefix, suffix, self.derivation, self.itype, self.role, self.ctype = self.KINDS[kind]
        self.instance = f"{prefix}:{col}:{suffix}"

    @property
    def ref(self):
        return f"[{self.ds.key}].[{self.instance}]"

    def column_attrs(self):
        attrs = {"datatype": self.ds.types[self.col], "name": f"[{self.col}]", "role": self.role, "type": self.ctype}
        if self.kind in ("sum", "avg"):
            attrs["aggregation"] = "Sum"
        return attrs

    def instance_attrs(self):
        return {"column": f"[{self.col}]", "derivation": self.derivation, "name": f"[{self.instance}]", "pivot": "key", "type": self.itype}


def sum_(ds, col): return Field(ds, col, "sum")
def avg_(ds, col): return Field(ds, col, "avg")
def cont_(ds, col): return Field(ds, col, "cont")
def month_c(ds, col): return Field(ds, col, "month_c")
def month_d(ds, col): return Field(ds, col, "month_d")

def dim_(ds, col):
    # Integers become ordinal headers (sorted numerically); everything else is nominal.
    return Field(ds, col, "ord" if ds.types[col] == "integer" else "nom")


sources = {
    "kpis": DataSource("kpis", "Headline KPIs", "headline_kpis.csv"),
    "monthly": DataSource("monthly", "Monthly sales", "monthly_gross_sales.csv"),
    "newret": DataSource("newret", "Existing vs new customers", "customers_existing_vs_new.csv"),
    "region": DataSource("region", "New customers by region", "new_customers_domestic_overseas.csv"),
    "cohort": DataSource("cohort", "Cohort retention", "cohort_heatmap.csv"),
    "repeat": DataSource("repeat", "90-day repeat rate", "repeat_rate_90_days.csv"),
    "segments": DataSource("segments", "RFM segment shares", "segment_shares_pct.csv"),
    "conc": DataSource("conc", "Revenue concentration", "revenue_concentration_pct.csv"),
    "churn": DataSource("churn", "Churn by days silent", "churn_by_silence.csv"),
    "rules": DataSource("rules", "Early-warning rules", "rule_table.csv"),
    "contacts": DataSource("contacts", "Contact list", "contact_list.csv"),
    "recs": DataSource("recs", "Recommendations", "recommendations_table.csv"),
}
{key: ds.types for key, ds in list(sources.items())[:3]}

{'kpis': {'gross_sales': 'real',
  'tracked_customers': 'integer',
  'repeat_rate': 'real',
  'sales_growth': 'real',
  'new_customer_change': 'real',
  'slipping_customers': 'integer',
  'slipping_revenue': 'real',
  'slipping_revenue_share': 'real',
  'quiet_customers': 'integer',
  'quiet_lifetime_sales': 'real',
  'flagged_customers': 'integer',
  'flagged_sales': 'real',
  'expected_loss': 'real',
  'top_tier_customers': 'integer',
  'top_tier_sales': 'real',
  'gross_sales_m': 'real',
  'slipping_revenue_m': 'real',
  'flagged_sales_k': 'real',
  'expected_loss_k': 'real',
  'top_tier_sales_k': 'real',
  'quiet_lifetime_sales_k': 'real'},
 'monthly': {'month': 'date', 'gross_sales': 'real', 'gross_sales_m': 'real'},
 'newret': {'month': 'date',
  'customer_type': 'string',
  'customers': 'integer'}}

## 2. Worksheets

In [3]:
def nest(fields):
    # Shelf expression: one field, or dimensions nested left to right with '/'.
    if not fields:
        return ""
    refs = [f.ref for f in fields]
    return refs[0] if len(refs) == 1 else "(" + " / ".join(refs) + ")"


def fmt(attr, value, field=None, scope=None):
    out = {"attr": attr, "value": value}
    if field is not None:
        out["field"] = field.ref
    if scope is not None:
        out["scope"] = scope
    return out


def add_style(parent, rules):
    # rules: {element: [format dicts or encoding elements]}
    # Tableau's schema requires a <style> element in every table, even an empty one.
    style = sub(parent, "style")
    for element, items in (rules or {}).items():
        rule = sub(style, "style-rule", {"element": element})
        for item in items:
            if isinstance(item, ET.Element):
                rule.append(item)
            else:
                sub(rule, "format", item)


def axis_fmt(field, scope, value):
    return {"attr": "text-format", "class": "0", "field": field.ref, "scope": scope, "value": value}


def axis_title(field, scope, text):
    return {"attr": "title", "class": "0", "field": field.ref, "scope": scope, "value": text}


worksheets = []  # (name, datasource, element)

def worksheet(name, ds, title_runs, rows=(), cols=(), mark="Automatic", color=None, text=None,
              table_rules=None, pane_rules=None):
    ws = ET.Element("worksheet", {"name": name})
    layout = sub(sub(ws, "layout-options"), "title")
    ft = sub(layout, "formatted-text")
    for run_text, run_color, bold in title_runs:
        attrs = {"fontcolor": run_color, "fontsize": "13" if bold else "10"}
        if bold:
            attrs["bold"] = "true"
        sub(ft, "run", attrs, run_text)

    table = sub(ws, "table")
    view = sub(table, "view")
    sub(sub(view, "datasources"), "datasource", {"caption": ds.caption, "name": ds.key})
    deps = sub(view, "datasource-dependencies", {"datasource": ds.key})
    used = [f for f in list(rows) + list(cols) + [color, text] if f is not None]
    seen_cols, seen_inst = set(), set()
    for f in used:
        if f.col not in seen_cols:
            sub(deps, "column", f.column_attrs())
            seen_cols.add(f.col)
    for f in used:
        if f.instance not in seen_inst:
            sub(deps, "column-instance", f.instance_attrs())
            seen_inst.add(f.instance)
    sub(view, "aggregation", {"value": "true"})

    add_style(table, table_rules)

    panes = sub(table, "panes")
    pane = sub(panes, "pane", {"selection-relaxation-option": "selection-relaxation-allow"})
    sub(sub(pane, "view"), "breakdown", {"value": "auto"})
    sub(pane, "mark", {"class": mark})
    enc = sub(pane, "encodings")
    if color is not None:
        sub(enc, "color", {"column": color.ref})
    if text is not None:
        sub(enc, "text", {"column": text.ref})
    if len(enc) == 0:
        pane.remove(enc)
    add_style(pane, pane_rules)

    sub(table, "rows", text=nest(list(rows)))
    sub(table, "cols", text=nest(list(cols)))
    worksheets.append((name, ds, ws))
    return ws

In [4]:
K = pd.read_csv(ROOT / "dashboard" / "tableau_data" / "kpis.csv").set_index("metric").value  # for text and titles
kp = sources["kpis"]

# ---------- headline tiles: one measure shown as large text ----------
def tile(name, col, title, number_format, color=INK):
    f = sum_(kp, col)
    worksheet(
        name, kp, [(title, INK2, False)], mark="Text", text=f,
        table_rules={"cell": [fmt("text-format", number_format, f), fmt("text-align", "center", f), fmt("font-size", "26", f), fmt("font-weight", "bold", f), fmt("color", color, f)]},
        pane_rules={"mark": [fmt("mark-labels-show", "true")]},
    )

PCT_LABEL = '*0"%"'  # for values already scaled to 0-100
M_FMT, K_FMT, INT_FMT, PCT_FMT = '*£0.0"M"', '*£#,##0"K"', "*#,##0", "*0%"

tile("T Gross sales", "gross_sales_m", "Gross sales, Dec 2009 to 9 Dec 2011", M_FMT)
tile("T Tracked customers", "tracked_customers", "Tracked customers", INT_FMT)
tile("T Repeat rate", "repeat_rate", "Repeat rate (2+ orders)", PCT_FMT)
tile("T Sales growth", "sales_growth", "Sales growth, Jan-Nov 2011 vs 2010", "*+0.0%;-0.0%")
tile("T New customers", "new_customer_change", "New customers, Jun-Nov 2011 vs 2010", "*+0%;-0%", color=ORANGE)
tile("T Slipping customers", "slipping_customers", "Slipping customers", INT_FMT)
tile("T Slipping revenue", "slipping_revenue_m", "Their net revenue to date", M_FMT)
tile("T Quiet VIPs", "quiet_customers", "Top-value customers silent 142+ days", INT_FMT, color=ORANGE)
tile("T Flagged customers", "flagged_customers", "Customers silent 90+ days", INT_FMT)
tile("T Flagged sales", "flagged_sales_k", "Their trailing 12m sales", K_FMT)
tile("T Expected loss", "expected_loss_k", "Expected loss at rule precision", K_FMT, color=ORANGE)
tile("T Top tier", "top_tier_customers", "Top-tier customers at risk", INT_FMT)
len(worksheets)

12

In [5]:
# ---------- Overview ----------
ms = sources["monthly"]
gross = sum_(ms, "gross_sales_m")
worksheet(
    "Monthly sales", ms,
    [("Gross sales by month", INK, True), ("   GBP millions, cancellations excluded; Dec 2011 is partial and left out", INK2, False)],
    rows=[gross], cols=[month_c(ms, "month")], mark="Bar",
    table_rules={"axis": [axis_title(gross, "rows", "Gross sales (GBP millions)"), axis_title(month_c(ms, "month"), "cols", "Month")]},
    pane_rules={"mark": [fmt("mark-color", BLUE)]},
)

nr = sources["newret"]
nr_type = Field(nr, "customer_type", "nom")
worksheet(
    "New vs returning", nr,
    [("Active customers: ", INK, True), ("\u25a0 Existing  ", BLUE, True), ("\u25a0 New", ORANGE, True)],
    rows=[sum_(nr, "customers")], cols=[month_c(nr, "month")], mark="Bar", color=nr_type,
    table_rules={"axis": [axis_title(sum_(nr, "customers"), "rows", "Customers"), axis_title(month_c(nr, "month"), "cols", "Month")]},
)

# ---------- Retention ----------
co = sources["cohort"]
ret = avg_(co, "retention")
worksheet(
    "Cohort heatmap", co,
    [("Cohort retention", INK, True), ("   share of each monthly cohort ordering again k months after the first purchase", INK2, False)],
    rows=[month_d(co, "cohort_month")], cols=[dim_(co, "months_since_first_purchase")], mark="Square", color=ret, text=ret,
    table_rules={"cell": [fmt("text-format", PCT_FMT, ret), fmt("font-size", "9", ret)]},
    pane_rules={"mark": [fmt("mark-labels-show", "true")]},
)

rp = sources["repeat"]
rate = sum_(rp, "repeat_pct")
worksheet(
    "Repeat 90d", rp,
    [("90-day repeat rate", INK, True), ("   new customers ordering again within 90 days (2009 Dec* overstated)", INK2, False)],
    rows=[rate], cols=[dim_(rp, "acquisition_period")], mark="Bar", text=rate,
    table_rules={"axis": [axis_title(rate, "rows", "Repeat within 90 days (%)"), axis_title(dim_(rp, "acquisition_period"), "cols", "Acquired")],
                 "cell": [fmt("text-format", PCT_LABEL, rate)],
                 "worksheet": [fmt("display-field-labels", "false", scope="cols")]},
    pane_rules={"mark": [fmt("mark-color", BLUE), fmt("mark-labels-show", "true")]},
)

rg = sources["region"]
rg_type = Field(rg, "region", "nom")
worksheet(
    "New customers by region", rg,
    [("New customers per month: ", INK, True), ("\u25a0 Domestic (UK)  ", BLUE, True), ("\u25a0 Overseas", ORANGE, True)],
    rows=[sum_(rg, "customers")], cols=[month_c(rg, "month")], mark="Bar", color=rg_type,
    table_rules={"axis": [axis_title(sum_(rg, "customers"), "rows", "New customers"), axis_title(month_c(rg, "month"), "cols", "Month")]},
)

# ---------- Segments ----------
sg = sources["segments"]
share = sum_(sg, "share_pct")
sg_measure = Field(sg, "measure", "nom")
worksheet(
    "RFM segments", sg,
    [("RFM segments: ", INK, True), ("\u25a0 Share of customers  ", BLUE, True), ("\u25a0 Share of net revenue", ORANGE, True)],
    rows=[dim_(sg, "revenue_rank"), dim_(sg, "segment"), sg_measure], cols=[share], mark="Bar", color=sg_measure, text=share,
    table_rules={
        "axis": [axis_title(share, "cols", "Share (%)")],
        "cell": [fmt("text-format", PCT_LABEL, share)],
        "label": [fmt("display", "false", dim_(sg, "revenue_rank")), fmt("display", "false", sg_measure)],
        "worksheet": [fmt("display-field-labels", "false", scope="rows")],
    },
    pane_rules={"mark": [fmt("mark-labels-show", "true")]},
)

cn = sources["conc"]
cum = sum_(cn, "sales_pct")
cust = cont_(cn, "customers_pct")
worksheet(
    "Concentration", cn,
    [("Revenue concentration", INK, True), ("   cumulative share of sales earned by the top share of customers", INK2, False)],
    rows=[cum], cols=[cust], mark="Line",
    table_rules={"axis": [axis_title(cum, "rows", "Share of sales (%)"), axis_title(cust, "cols", "Share of customers, ranked by sales (%)")]},
    pane_rules={"mark": [fmt("mark-color", BLUE)]},
)

# ---------- Churn risk ----------
ch = sources["churn"]
churned = sum_(ch, "churned_pct")
worksheet(
    "Churn by silence", ch,
    [("Churn rises with silence", INK, True), ("   no purchase in the next 161 days, by days since last purchase", INK2, False)],
    rows=[churned], cols=[dim_(ch, "sort_order"), dim_(ch, "days_silent")], mark="Bar", text=churned,
    table_rules={"axis": [axis_title(churned, "rows", "Churned (%)"), axis_title(dim_(ch, "days_silent"), "cols", "Days since last purchase")],
                 "cell": [fmt("text-format", PCT_LABEL, churned)],
                 "label": [fmt("display", "false", dim_(ch, "sort_order"))],
                 "worksheet": [fmt("display-field-labels", "false", scope="cols")]},
    pane_rules={"mark": [fmt("mark-color", ORANGE), fmt("mark-labels-show", "true")]},
)

def list_table(name, key, title, subtitle, widths=None):
    ds = sources[key]
    fields = [dim_(ds, c) for c in ds.columns]
    ghost = sum_(ds, "Rank")  # invisible text mark (format ;;;) so Tableau shows no "Abc" placeholder
    rules = {"cell": [fmt("text-format", "*;;;", ghost)]}
    header_rules = [fmt("width", str(w), Field(ds, c, "ord" if ds.types[c] == "integer" else "nom")) for c, w in (widths or {}).items()]
    if header_rules:
        rules["header"] = header_rules
    worksheet(name, ds, [(title, INK, True), (subtitle, INK2, False)], rows=fields, mark="Text", text=ghost, table_rules=rules)

list_table("Rule back-test", "rules", "Early-warning rules, back-tested", "   precision = share of flagged who churned; recall = share of churners flagged (at other cutoffs recall is 45-76%)", widths={"Rule": 270, "Churner revenue caught": 170, "Flagged": 80})
list_table("Contact list", "contacts", "Top-value customers to contact first", "   highest trailing 12m sales among customers silent 90+ days (as of 10 Dec 2011)", widths={"Days silent": 95, "Own avg gap days": 140, "Order days": 100, "Trailing 12m sales": 150, "Early flag": 90})
list_table("Recommendations", "recs", "Recommendations, ranked by value for effort", "   annual impact under low / mid / high scenarios (scenario percentages are assumptions)", widths={"Action": 640, "Effort": 120, "Measured base": 130})
[name for name, _, _ in worksheets]

['T Gross sales',
 'T Tracked customers',
 'T Repeat rate',
 'T Sales growth',
 'T New customers',
 'T Slipping customers',
 'T Slipping revenue',
 'T Quiet VIPs',
 'T Flagged customers',
 'T Flagged sales',
 'T Expected loss',
 'T Top tier',
 'Monthly sales',
 'New vs returning',
 'Cohort heatmap',
 'Repeat 90d',
 'New customers by region',
 'RFM segments',
 'Concentration',
 'Churn by silence',
 'Rule back-test',
 'Contact list',
 'Recommendations']

## 3. Dashboards

Positions are written in pixels and converted to Tableau's 0-100000 zone units. Each dashboard is a fixed 1280 px wide canvas.

**Design**
- A navy **header banner** on every dashboard carries a one-line takeaway (the "so what") and the dashboard name.
- A light gray page with **white bordered cards**; every card has 6 px of outer padding, which makes the gutters.
- Headline numbers sit in a tile row on top, charts below, and a plain-language **insight card** where a chart needs context.

In [6]:
DASH_W = 1280
PAGE_BG, CARD_BG, CARD_BORDER = "#f3f4f6", "#ffffff", "#dfe2e8"
NAVY, BANNER_SUB = "#1f2a44", "#c9d1e3"
dashboards = []  # (name, height, element, [sheet names])

def run(text, size=None, color=None, bold=False):
    attrs = {}
    if size:
        attrs["fontsize"] = str(size)
    if color:
        attrs["fontcolor"] = color
    if bold:
        attrs["bold"] = "true"
    return text, attrs

def zone_style(zone, formats):
    style = sub(zone, "zone-style")
    for attr, value in formats.items():
        sub(style, "format", {"attr": attr, "value": value})

CARD = {"background-color": CARD_BG, "border-color": CARD_BORDER, "border-style": "solid", "border-width": "1", "margin": "6"}
BANNER = {"background-color": NAVY, "border-style": "none", "border-width": "0", "margin": "0"}

def dashboard(name, height, items):
    # items: (kind, payload, x, y, w, h) with kind "sheet" (payload = sheet name), "text" (card) or "banner" (header)
    dash = ET.Element("dashboard", {"name": name})
    sub(dash, "size", {"maxheight": height, "maxwidth": DASH_W, "minheight": height, "minwidth": DASH_W})
    used = sorted({ds.key for sheet, ds, _ in worksheets for it in items if it[0] == "sheet" and it[1] == sheet})
    dss = sub(dash, "datasources")
    for key in used:
        sub(dss, "datasource", {"caption": sources[key].caption, "name": key})

    zones = sub(dash, "zones")
    outer = sub(zones, "zone", {"h": 100000, "id": 1, "type": "layout-basic", "w": 100000, "x": 0, "y": 0})
    zone_id = 1
    sheets = []
    for item in items:
        zone_id += 1
        kind, payload, x, y, w, h = item
        geometry = {"h": round(h / height * 100000), "id": zone_id, "w": round(w / DASH_W * 100000),
                    "x": round(x / DASH_W * 100000), "y": round(y / height * 100000)}
        if kind == "sheet":
            z = sub(outer, "zone", {**geometry, "name": payload, "show-title": "true"})
            zone_style(z, CARD)
            sheets.append(payload)
        else:
            z = sub(outer, "zone", {**geometry, "type": "text"})
            ft = sub(z, "formatted-text")
            for text, attrs in payload:
                sub(ft, "run", attrs, text)
            zone_style(z, BANNER if kind == "banner" else CARD)
    zone_style(outer, {"background-color": PAGE_BG})  # Tableau's schema wants zone-style after the child zones
    dashboards.append((name, height, dash, sheets))

W = DASH_W

def banner(headline, tab):
    return ("banner", [run("  " + headline + "\n", 17, "#ffffff", True),
                       run("  Online Retail II  \u00b7  " + tab + "  \u00b7  Dec 2009 to 9 Dec 2011", 9, BANNER_SUB)], 0, 0, W, 64)

findings_runs = [
    run("Key insights\n", 14, INK, True),
    run(f"\u2022 Sales are flat ({K['yoy'] * 100:+.1f}% on the same months a year earlier), but not because customers leave after one order: {K['repeat_rate'] * 100:.0f}% reorder.\n", 11, INK2),
    run(f"\u2022 New-customer acquisition fell {abs(K['new_cust_change']) * 100:.0f}% (Jun-Nov 2011 vs 2010). Cohort quality has been stable since mid-2010, so volume is the part to investigate.\n", 11, INK2),
    run(f"\u2022 Revenue is concentrated: the top 10% of customers bring {K['top10_share'] * 100:.0f}% of sales; RFM Champions ({K['champ_cust_share'] * 100:.0f}% of customers) bring {K['champ_rev_share'] * 100:.0f}%.\n", 11, INK2),
    run(f"\u2022 Silence is the churn signal, not slowing gaps or cancellations. A 90-day rule flags {K['flagged_customers']:,.0f} customers worth \u00a3{K['flagged_rev'] / 1e3:,.0f}K in trailing sales.\n", 11, INK2),
    run(f"\u2022 Leaks: cancellations are {K['cancel_rate'] * 100:.1f}% of gross sales; {K['guest_share'] * 100:.0f}% of sales have no customer ID.", 11, INK2),
]

retention_notes = [
    run("What to notice\n", 14, INK, True),
    run("\u2022 About 20% of every 2010-2011 cohort orders again in month 1; the 90-day repeat rate fell from 45% (2010 H1) to about 41% and then held.\n", 11, INK2),
    run("\u2022 Activity is seasonal: cohorts light up again when the Sep-Nov peak returns.\n", 11, INK2),
    run(f"\u2022 New customers fell {abs(K['new_cust_change']) * 100:.0f}% (Jun-Nov 2011 vs 2010); UK arrivals dropped 43%.\n", 11, INK2),
    run("\u2022 The 2009 Dec cohort is overstated: it includes people who bought before the data starts.", 11, INK2),
]

dashboard("Overview", 1010, [
    banner(f"Sales are flat while new-customer acquisition fell {abs(K['new_cust_change']) * 100:.0f}%", "Overview"),
    ("sheet", "T Gross sales", 0, 64, 360, 150), ("sheet", "T Tracked customers", 360, 64, 230, 150), ("sheet", "T Repeat rate", 590, 64, 230, 150),
    ("sheet", "T Sales growth", 820, 64, 230, 150), ("sheet", "T New customers", 1050, 64, 230, 150),
    ("sheet", "Monthly sales", 0, 214, W, 370),
    ("sheet", "New vs returning", 0, 584, 700, 416), ("text", findings_runs, 700, 584, 580, 416),
])

dashboard("Retention", 1090, [
    banner("Cohort quality is stable since mid-2010; acquisition volume is what fell", "Retention"),
    ("sheet", "Cohort heatmap", 0, 64, W, 650),
    ("sheet", "Repeat 90d", 0, 714, 440, 366), ("sheet", "New customers by region", 440, 714, 480, 366), ("text", retention_notes, 920, 714, 360, 366),
])

dashboard("Segments", 990, [
    banner(f"{K['champ_cust_share'] * 100:.0f}% of customers bring {K['champ_rev_share'] * 100:.0f}% of revenue", "Segments"),
    ("sheet", "RFM segments", 0, 64, W, 560),
    ("sheet", "Concentration", 0, 624, 640, 356),
    ("sheet", "T Slipping customers", 640, 624, 214, 160), ("sheet", "T Slipping revenue", 854, 624, 214, 160), ("sheet", "T Quiet VIPs", 1068, 624, 212, 160),
    ("text", [run("Who is slipping away?\n", 13, INK, True),
              run("Need Attention, About to Sleep, At Risk and Can't Lose Them. 142 days is twice the 71-day upper end of a typical reorder gap. Many At Risk customers have been silent 6-13 months and may already be lost.", 11, INK2)], 640, 784, 640, 196),
])

dashboard("Churn risk", 1040, [
    banner("Silence is the signal: contact customers quiet for 90+ days, biggest first", "Churn risk"),
    ("sheet", "T Flagged customers", 0, 64, 320, 150), ("sheet", "T Flagged sales", 320, 64, 320, 150),
    ("sheet", "T Expected loss", 640, 64, 320, 150), ("sheet", "T Top tier", 960, 64, 320, 150),
    ("sheet", "Churn by silence", 0, 214, 560, 350), ("sheet", "Rule back-test", 560, 214, 720, 350),
    ("sheet", "Contact list", 0, 564, W, 466),
])

dashboard("Actions", 650, [
    banner("Four moves, sized under low, mid and high scenarios", "Actions"),
    ("sheet", "Recommendations", 0, 64, W, 340),
    ("text", [run("What this data cannot tell you\n", 13, INK, True),
              run("\u2022 Why acquisition fell (there is no channel or marketing data).\n", 11, INK2),
              run("\u2022 What a win-back campaign will really recover: run a small test with a holdout group before scaling.\n", 11, INK2),
              run("\u2022 Causes: product mix adds little to churn prediction (notebook 11), but nothing here shows what changes customer behaviour.", 11, INK2)], 0, 404, W, 236),
])
[(n, h, s) for n, h, _, s in dashboards][:2]

[('Overview',
  1010,
  ['T Gross sales',
   'T Tracked customers',
   'T Repeat rate',
   'T Sales growth',
   'T New customers',
   'Monthly sales',
   'New vs returning']),
 ('Retention',
  1090,
  ['Cohort heatmap', 'Repeat 90d', 'New customers by region'])]

## 4. Assemble and write the workbook

In [7]:
def window_for_worksheet(parent, name):
    win = sub(parent, "window", {"class": "worksheet", "name": name})
    cards = sub(win, "cards")
    left = sub(sub(cards, "edge", {"name": "left"}), "strip", {"size": 160})
    for card in ("pages", "filters", "marks"):
        sub(left, "card", {"type": card})
    top = sub(cards, "edge", {"name": "top"})
    for card in ("columns", "rows", "title"):
        sub(sub(top, "strip", {"size": 2147483647}), "card", {"type": card})
    return win

workbook = ET.Element("workbook", {
    "locale": "en_US", "source-build": "0.0.0 (0000.0.0.0)", "source-platform": "win", "version": "18.1",
    "xmlns:user": "http://www.tableausoftware.com/xml/user",
})
sub(workbook, "preferences")
dss_el = sub(workbook, "datasources")
for ds in sources.values():
    dss_el.append(ds.element())
ws_el = sub(workbook, "worksheets")
for _, _, ws in worksheets:
    ws_el.append(ws)
db_el = sub(workbook, "dashboards")
for _, _, dash, _ in dashboards:
    db_el.append(dash)

windows = sub(workbook, "windows", {"source-height": 32})
for i, (name, _, _, sheets) in enumerate(dashboards):
    attrs = {"class": "dashboard", "name": name}
    if i == 0:
        attrs["maximized"] = "true"
    win = sub(windows, "window", attrs)
    vps = sub(win, "viewpoints")
    for s in sheets:
        sub(sub(vps, "viewpoint", {"name": s}), "zoom", {"type": "entire-view"})
    sub(win, "active", {"id": -1})
for name, _, _ in worksheets:
    window_for_worksheet(windows, name)

ET.indent(workbook, space="  ")
OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
ET.ElementTree(workbook).write(OUT_FILE, encoding="utf-8", xml_declaration=True)
print(f"wrote {OUT_FILE.name}: {OUT_FILE.stat().st_size / 1e3:.0f} KB | {len(sources)} data sources, {len(worksheets)} worksheets, {len(dashboards)} dashboards")

wrote retail_dashboard_generated.twb: 111 KB | 12 data sources, 23 worksheets, 5 dashboards


## 5. Package as .twbx

A `.twbx` is a zip file holding the workbook plus its data, so it opens on any computer without the project folder. The layout follows Tableau's own packaged samples: `retail_dashboard_generated.twb` at the root and the CSV files under `Data/retail_dashboard/`, with every text-file connection rewritten to that relative folder.

In [8]:
import io
import zipfile

PKG_DIR = "Data/retail_dashboard"
TWBX_FILE = OUT_FILE.with_suffix(".twbx")

packaged = ET.parse(OUT_FILE)
for conn in packaged.getroot().iter("connection"):
    if conn.get("class") == "textscan":
        conn.set("directory", PKG_DIR)       # relative to the package, not an absolute path on this computer
buffer = io.BytesIO()
packaged.write(buffer, encoding="utf-8", xml_declaration=True)

with zipfile.ZipFile(TWBX_FILE, "w", zipfile.ZIP_DEFLATED) as pkg:
    pkg.writestr(OUT_FILE.name, buffer.getvalue())
    for ds in sources.values():
        pkg.write(DATA_DIR / ds.filename, f"{PKG_DIR}/{ds.filename}")

# Check the package: every data source file is inside, and no connection still points at an absolute path.
with zipfile.ZipFile(TWBX_FILE) as pkg:
    names = set(pkg.namelist())
    twb_inside = ET.fromstring(pkg.read(OUT_FILE.name))
missing = [ds.filename for ds in sources.values() if f"{PKG_DIR}/{ds.filename}" not in names]
dirs = {c.get("directory") for c in twb_inside.iter("connection") if c.get("class") == "textscan"}
assert not missing and dirs == {PKG_DIR}, (missing, dirs)
print(f"wrote {TWBX_FILE.name}: {TWBX_FILE.stat().st_size / 1e3:.0f} KB | {len(names)} files | connections point to '{PKG_DIR}'")

wrote retail_dashboard_generated.twbx: 15 KB | 13 files | connections point to 'Data/retail_dashboard'


## 6. Check the file

In [9]:
# Re-parse the file and cross-check every reference: zones -> worksheets, fields -> columns -> CSV headers.
tree = ET.parse(OUT_FILE).getroot()
sheet_names = {w.get("name") for w in tree.find("worksheets")}
problems = []

for dash in tree.find("dashboards"):
    for zone in dash.iter("zone"):
        if zone.get("name") and zone.get("name") not in sheet_names:
            problems.append(f"dashboard {dash.get('name')}: unknown sheet {zone.get('name')}")

import re
for ws in tree.find("worksheets"):
    view = ws.find("table/view")
    deps = view.find("datasource-dependencies")
    ds_key = deps.get("datasource")
    csv_cols = set(sources[ds_key].columns)
    defined = {c.get("name") for c in deps.findall("column")}
    instances = {i.get("name") for i in deps.findall("column-instance")}
    for c in defined:
        if c.strip("[]") not in csv_cols:
            problems.append(f"{ws.get('name')}: column {c} not in {sources[ds_key].filename}")
    for shelf in ("rows", "cols"):
        for inst in re.findall(r"\]\.\[([^\]]+)\]", ws.findtext(f"table/{shelf}") or ""):
            if f"[{inst}]" not in instances:
                problems.append(f"{ws.get('name')}: {shelf} uses undefined instance {inst}")

print("problems:", problems if problems else "none")
assert not problems

problems: none


## 7. Open it

Double-click `dashboard/tableau/retail_dashboard_generated.twb`, or in Tableau Desktop use File > Open. Five dashboards appear as tabs along the bottom, followed by the individual worksheets.

If Tableau reports an error or a chart looks different from the HTML dashboard, note the message and the sheet name; the worksheet definitions in section 2 are the place to adjust.